In [1]:
import numpy as np

def rbf_kernel(xa, xb, sigma_f=1.0, ell=1.0):
    xa = xa[:, None]
    xb = xb[None, :]
    sqdist = (xa - xb)**2
    return sigma_f**2 * np.exp(-0.5 * sqdist / ell**2)

# --- toy data ---
rng = np.random.default_rng(0)
X = np.array([-3.0, -2.0, -1.0, 0.3, 1.2, 2.7])
f_true = lambda x: np.sin(1.5*x)
sigma_n = 0.2
y = f_true(X) + sigma_n * rng.normal(size=X.shape)

# --- GP hyperparameters (try changing these) ---
sigma_f = 1.0
ell = 1.0

# --- prediction grid ---
Xstar = np.linspace(-4, 4, 300)

K = rbf_kernel(X, X, sigma_f=sigma_f, ell=ell)
Ky = K + sigma_n**2 * np.eye(len(X))
Ks = rbf_kernel(X, Xstar, sigma_f=sigma_f, ell=ell)
Kss = rbf_kernel(Xstar, Xstar, sigma_f=sigma_f, ell=ell)

# stable solve instead of explicit inverse
L = np.linalg.cholesky(Ky)
alpha = np.linalg.solve(L.T, np.linalg.solve(L, y))

mu = Ks.T @ alpha
v = np.linalg.solve(L, Ks)
cov = Kss - v.T @ v
std = np.sqrt(np.maximum(np.diag(cov), 0.0))

# mu = posterior mean, std = posterior std at Xstar
print(mu[:5], std[:5])


[0.7347058  0.75228036 0.7696435  0.78675779 0.80358516] [0.74671684 0.73276009 0.71837583 0.70357523 0.68837151]
